# 06 Context ablation
Build a request from named blocks, remove one at a time, then test compaction and memory.

Step 1: set up the kit

In [ ]:
# Setup: in Colab, clone the kit first. Locally, find the kit folder. Change REPO_URL to your fork (and BRANCH).
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/arpitRanjan/atp-builder-kit"
BRANCH = "main"  # change to your branch name if you are working on a branch
if "google.colab" in sys.modules and not Path("kit/llm.py").exists():
    if not Path("atp-builder-kit").exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL], check=True)
    os.chdir("atp-builder-kit")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-genai==3.3.0", "openai==3.51.0"], check=True)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kit" / "llm.py").exists())
sys.path.insert(0, str(ROOT))
CORPUS = Path(os.environ.get("CORPUS_DIR", ROOT / "corpus"))
OUTPUT = Path(os.environ.get("KIT_OUTPUT_DIR", ROOT / "stage-06"))
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Kit folder found:", ROOT.name)

Step 2: the named blocks

In [ ]:
import json, re
import pandas as pd
from kit.llm import call_llm

BLOCKS = {
    "system": "You are the Quillnote help assistant. Answer in at most 2 sentences.",
    "examples": ("Q: How do I turn on dark mode?\nA: Open Settings, then Appearance, and pick Dark. [qn-06]\n\n"
                 "Q: Can I get a deleted note back?\nA: Open Trash, pick the note and select Restore. [qn-07]"),
    "tools": ("- search_docs(query): look up an answer in the Quillnote help pages.\n"
              "- draft_issue(title, body): report a bug or broken feature to the engineering team.\n"
              "- open_billing_case(email): send a payment, charge or refund problem to the billing team."),
    "history": ("User: Hi, I am Priya and my workspace is Bluefin.\nAssistant: Hi Priya, how can I help?\n"
                "User: I am on the Plus plan.\nAssistant: Thanks, noted.\n"
                "User: I use Quillnote on an Android phone.\nAssistant: Got it."),
    "instructions": ("# AGENTS.md for the help assistant\n"
                     "- If the question is about refunds, charges or billing, say: Please contact billing@quillnote.example.\n"
                     "- If the question is about deleting a user's data, say: Only workspace admins can do this.\n"
                     "- Never say 'guaranteed'."),
}

Step 3: the help pages each request carries (never removed)

In [ ]:
pages = []
for path in sorted(CORPUS.glob("*.md")):
    if path.name.lower() != "readme.md":
        _, head, body = path.read_text().split("---", 2)
        meta = {k.strip(): v.strip() for k, v in (line.split(":", 1) for line in head.strip().splitlines())}
        pages.append({"id": meta["id"], "text": " ".join(body.split()[:60])})

def top_pages(question, n=2):
    q = set(re.findall(r"[a-z]+", question.lower()))
    ranked = sorted(pages, key=lambda p: -len(q & set(re.findall(r"[a-z]+", p["text"].lower()))))
    return "\n".join(f"[{p['id']}] {p['text']}" for p in ranked[:n])

def build_request(blocks, question):
    # Every block goes in one prompt, so removing one is one line. In an app, system goes in the system field.
    parts = [f"## {name.upper()}\n{text}" for name, text in blocks.items()]
    return "\n\n".join(parts + [f"## PAGES\n{top_pages(question)}", f"## QUESTION\n{question}"])

print(build_request(BLOCKS, "How do I recover a deleted note?")[:600])

Step 4: a probe set where each probe needs one block

In [ ]:
PROBES = [  # (id, question, check, value, the block it needs)
    ("p01", "How do I recover a deleted note?", "regex", r"\[qn-\d+\]", "examples"),
    ("p02", "How do I export a note as PDF?", "regex", r"\[qn-\d+\]", "examples"),
    ("p03", "How do I fix sync problems?", "regex", r"\[qn-\d+\]", "examples"),
    ("p04", "Which tool would you call to report that PDF export is broken? Reply with the tool name only.",
     "equals", "draft_issue", "tools"),
    ("p05", "Which tool would you call for a refund problem? Reply with the tool name only.",
     "equals", "open_billing_case", "tools"),
    ("p06", "Which tool would you call to look up how tags work? Reply with the tool name only.",
     "equals", "search_docs", "tools"),
    ("p07", "Which plan am I on?", "contains", "Plus", "history"),
    ("p08", "What phone do I use Quillnote on?", "contains", "Android", "history"),
    ("p09", "Which workspace am I in?", "contains", "Bluefin", "history"),
    ("p10", "Can I get a refund for a double charge?", "contains", "billing@quillnote.example", "instructions"),
    ("p11", "How do I delete a user's data?", "contains", "admins", "instructions"),
    ("p12", "Is PDF export guaranteed to keep my images?", "not_contains", "guaranteed", "instructions"),
    ("p13", "How do I share a notebook with my team?", "max_sentences", "2", "system"),
    ("p14", "How do I switch to dark mode?", "max_sentences", "2", "system"),
]

def passes(text, check, value):
    text = text.strip()
    if check == "regex":
        return re.search(value, text) is not None
    if check == "equals":
        return text.strip("`. ").lower() == value
    if check == "contains":
        return value.lower() in text.lower()
    if check == "not_contains":
        return value.lower() not in text.lower()
    if check == "max_sentences":
        return len([s for s in re.split(r"(?<=[.!?])\s+", text) if s]) <= int(value)
    raise ValueError(check)

Step 5: remove one block at a time

In [ ]:
variants = {"all blocks": BLOCKS} | {f"no {name}": {k: v for k, v in BLOCKS.items() if k != name} for name in BLOCKS}
rows = []
for variant, blocks in variants.items():
    for pid, question, check, value, needs in PROBES:
        r = call_llm(build_request(blocks, question), max_output_tokens=1000)
        rows.append({"variant": variant, "probe": pid, "needs": needs, "passed": passes(r["text"], check, value),
                     "input_tokens": r["input_tokens"]})
ablation = pd.DataFrame(rows)
summary = ablation.groupby("variant", sort=False).agg(pass_rate=("passed", "mean"), input_tokens=("input_tokens", "mean"))
summary["failed_probes"] = [", ".join(ablation[(ablation.variant == v) & ~ablation.passed].probe) for v in summary.index]
summary.round(2).to_csv(OUTPUT / "context_ablation.csv")  # the context ablation table
summary.round(2)

Step 6: a long session with three constraints planted early

In [ ]:
CONSTRAINTS = {  # key phrase to look for: the constraint as the user said it
    "British": "Always use British spelling in your replies.",
    "email": "Never put my email address in a ticket.",
    "20 dollars": "My budget for add-ons is 20 dollars a month.",
}
questions = [json.loads(line)["question"] for line in open(CORPUS / "questions.jsonl")]
session = [f"User: {c}\nAssistant: Understood." for c in CONSTRAINTS.values()]
session += [f"User: {q}\nAssistant: Here is how, step by step, from the help pages." for q in questions]
print(len(session), "turns,", len("\n".join(session).split()), "words")

Step 7: summarize and restart, then check which constraints survive

In [ ]:
summary = call_llm("Summarize this session in at most 80 words so a new assistant can continue it. "
                   "Keep every rule and fact the user gave.\n\n" + "\n".join(session), max_output_tokens=1000)["text"]
print("Summary:", summary, "\n")
RESTART_CHECKS = [("British", "What spelling do I want you to use?"),
                  ("email", "What do I never want in a ticket?"),
                  ("20 dollars", "What budget do I have for add-ons?")]

def restart_check(history):
    out = []
    for key, question in RESTART_CHECKS:
        text = call_llm(build_request({"history": history}, question), max_output_tokens=1000)["text"]
        out.append({"constraint": CONSTRAINTS[key], "in_summary": key.lower() in history.lower(),
                    "answered_after_restart": key.split()[0].lower() in text.lower()})
    return pd.DataFrame(out)

compaction = restart_check("User: " + summary).assign(history="summary only")
compaction

Step 8: the fix: pin the constraints so compaction never touches them

In [ ]:
pinned = "\n".join(f"User: {c}" for c in CONSTRAINTS.values())
fixed = restart_check(pinned + "\nUser: " + summary).assign(history="pinned constraints + summary")
pd.concat([compaction, fixed]).to_csv(OUTPUT / "compaction_check.csv", index=False)  # the compaction check output
fixed

Step 9: plant one wrong memory note, then run a consolidation pass

In [ ]:
NOTES = ["User prefers British spelling.", "User is on the Plus plan.", "User uses Quillnote on Android.",
         "User uses Quillnote on Android.", "User is on the Free plan."]
WRONG = "User is on the Free plan."  # planted: the account system says Plus
reply = call_llm("Consolidate these memory notes: merge duplicates and drop any note that is wrong or "
                 "contradicted. Return the kept notes as a JSON list of strings.\n" + "\n".join(f"- {n}" for n in NOTES),
                 max_output_tokens=1000)["text"]
kept = json.loads(reply[reply.find("["): reply.rfind("]") + 1])
print("Kept:", kept)
print("Wrong note removed by the model pass:", WRONG not in kept)

Step 10: check memory against the system of record, in code

In [ ]:
ACCOUNT = {"plan": "Plus"}  # the account system is the truth, not the notes

def consistent(note):
    plan = re.search(r"on the (\w+) plan", note)
    return plan is None or plan.group(1) == ACCOUNT["plan"]

checked = [n for n in kept if consistent(n)]
print("After the code check:", checked)
print("Wrong note removed:", WRONG not in checked)